# 01 · Your first LLM call

**AI Engineering Bootcamp · Module 1**

In this notebook you will:

1. Call an LLM from Python using the OpenAI SDK
2. Read the answer, the token usage and the latency
3. Send a conversation with system, user and assistant messages
4. See what temperature does
5. Switch from Gemini to Groq by changing two lines
6. Compare a weak prompt with a strong prompt

**Before you start**

- Open this folder in VS Code and select your `.venv` as the kernel (top right, *Select Kernel*).
- Copy `.env.example` to `.env` and paste in your own keys.
- Run a cell with **Shift + Enter**.

## Step 0 · Install packages (only needed once)

This installs into the same environment as the notebook's kernel.

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1 · Load your keys

In [2]:
import os
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads the .env file in this folder

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")

# Never print the key itself, only whether it was found
print("Gemini key found:", bool(os.getenv("GEMINI_API_KEY")))
print("Groq key found:  ", bool(os.getenv("GROQ_API_KEY")))

Gemini key found: True
Groq key found:   True


## Step 2 · Make your first call

In [3]:
client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url=GEMINI_BASE_URL,
)

response = client.chat.completions.create(
    model=GEMINI_MODEL,
    messages=[
        {"role": "system", "content": "You are a concise tutor."},
        {"role": "user", "content": "Explain tokens in two lines."},
    ],
)

print(response.choices[0].message.content)

Tokens are the basic units of text—such as words, sub-words, or characters—that AI models read and process. 
Think of them as the bite-sized building blocks an AI uses to understand and generate language.


**If you get an error**

| Error | Usual cause |
|---|---|
| 401 | Wrong or missing key. Check `.env` and re-run Step 1. |
| 404 | Wrong model name. Copy the current name from the provider's model list. |
| 429 | Rate limit. Wait a minute, or switch to Groq (Step 6). |

## Step 3 · Read the response: text, tokens and time

In [4]:
start = time.perf_counter()
response = client.chat.completions.create(
    model=GEMINI_MODEL,
    messages=[{"role": "user", "content": "Give me 3 tips for writing clean Python."}],
)
latency = time.perf_counter() - start

print(response.choices[0].message.content)
print("-" * 40)
print("model:", response.model)
usage = response.usage
if usage:
    print("tokens in: ", usage.prompt_tokens)
    print("tokens out:", usage.completion_tokens)
print(f"latency:    {latency:.2f}s")

Here are three impactful tips for writing clean, readable, and maintainable Python:

---

### 1. Embrace "Pythonic" Idioms
Writing clean Python means not treating it like C++ or Java. Leverage Python’s built-in syntax to make your code more expressive and reduce boilerplate.

* **Avoid `range(len(...))`:** Iterate directly over collections, or use `enumerate()` if you need the index.
  ```python
  # Un-Pythonic
  for i in range(len(users)):
      print(i, users[i])

  # Clean
  for i, user in enumerate(users):
      print(i, user)
  ```
* **Use Context Managers (`with`):** Always use `with` when dealing with files, locks, or network connections to guarantee proper resource cleanup.
* **Leverage List Comprehensions responsibly:** Use them for simple transformations, but fall back to standard `for` loops if the logic exceeds one line of conditions.

---

### 2. Use Type Hints and `dataclasses` for Structure
Python is dynamically typed, but using modern typing makes code self-documenting,

## Step 4 · Messages and roles

The model has **no memory between calls**. A conversation is just a list of messages you send every time.
Here we keep the list, add the model's reply, then ask a follow-up question.

In [5]:
messages = [
    {"role": "system", "content": "You are a friendly Python tutor. Keep answers short."},
    {"role": "user", "content": "What is a list in Python?"},
]

reply = client.chat.completions.create(model=GEMINI_MODEL, messages=messages)
answer = reply.choices[0].message.content
print("Assistant:", answer)

# Add the reply and a follow-up question, then send the whole list again
messages.append({"role": "assistant", "content": answer})
messages.append({"role": "user", "content": "Show me an example."})

reply = client.chat.completions.create(model=GEMINI_MODEL, messages=messages)
print("\nAssistant:", reply.choices[0].message.content)
print("\nMessages sent this time:", len(messages))

Assistant: A **list** in Python is an ordered collection of items that you can change (it's "mutable"). Think of it like a shopping list or a to-do list!

You create one by putting items inside square brackets `[]`, separated by commas:

```python
fruits = ["apple", "banana", "cherry"]
numbers = [1, 2, 3, 4]
```

Lists can hold text, numbers, or even a mix of different data types!

Assistant: Here is a quick example!

```python
# 1. Create a list of pets
pets = ["dog", "cat", "fish"]

# 2. Access the first item (Python starts counting at 0)
print(pets[0])  # Output: dog

# 3. Add a new pet to the list
pets.append("parrot")

# 4. See the updated list
print(pets)  # Output: ['dog', 'cat', 'fish', 'parrot']
```

Messages sent this time: 4


## Step 5 · Temperature

Lower temperature gives more focused, repeatable answers. Higher temperature gives more varied ones.

In [6]:
question = [{"role": "user", "content": "Suggest a name for a study-planner app. Reply with the name only."}]

for temp in [0.2, 0.2, 0.9, 0.9]:
    r = client.chat.completions.create(model=GEMINI_MODEL, messages=question, temperature=temp)
    print(f"temperature {temp}: {r.choices[0].message.content.strip()}")

temperature 0.2: ScholarSync
temperature 0.2: StudyFlow
temperature 0.9: StudySync
temperature 0.9: ScholarSync


## Step 6 · Switch provider: Gemini to Groq

Same code. Only the key, `base_url` and model change.

In [8]:
groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url=GROQ_BASE_URL,
)

start = time.perf_counter()
r = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[{"role": "user", "content": "Explain tokens in two lines."}],
)
print(r.choices[0].message.content)
print(f"\nGroq latency: {time.perf_counter() - start:.2f}s")

Tokens are the smallest units of text that a language model processes—often words or subwords.  
They let the model handle diverse inputs efficiently by breaking sentences into these manageable pieces.

Groq latency: 0.55s


## Step 7 · Weak prompt vs strong prompt

In [9]:
text = """Git is a version control system. It tracks changes to files over time,
lets several people work on the same project, and makes it possible to go back
to an earlier version when something breaks."""

weak = "Summarize this."
strong = (
    "Summarize the text below in 3 bullet points for a beginner. "
    "Use simple English. If the text is not about software, reply 'Off topic'."
)

for name, prompt in [("WEAK", weak), ("STRONG", strong)]:
    r = client.chat.completions.create(
        model=GEMINI_MODEL,
        messages=[{"role": "user", "content": f"{prompt}\n\n{text}"}],
    )
    print(f"--- {name} ---")
    print(r.choices[0].message.content, "\n")

--- WEAK ---
Git is a version control system that tracks file changes over time, enables collaboration on a project, and allows users to revert to earlier versions if something breaks. 

--- STRONG ---
* Git keeps track of all the changes you make to your files over time.
* It lets many people work together on the same project without getting in each other's way.
* It allows you to go back to an older version if you make a mistake or something breaks. 



## Try it yourself

1. Change the system message in Step 2 to *"You explain everything using cricket examples."* What changes?
2. In Step 4, ask a third follow-up question. How many tokens in does it use now, and why?
3. In Step 7, replace `text` with a paragraph that is **not** about software. Does the strong prompt say *Off topic*?